# RSNA Knee: Top-Scorer ResNet50d Raptor 5-Fold Training

Trained on top of the verified 0.950 top-scorer Raptor architecture:
1. Native 3-slice windowing (8 windows x 3 consecutive slices)
2. 0.8 center field-of-view crop (`c08`) focusing high resolution on articular cartilage and cruciate ligaments
3. Backbone: `resnet50d` with exact ImageNet tensor normalization
4. Multi-finding 12-head attention pooling (`RaptorClassifier`) allowing independent slice weighting per pathology
5. Multi-tier Focal Loss with calibrated class weights (2.2 Lateral OA, 1.8 ACL/MCL, 1.5 Meniscus)
6. Stratified 5-Fold cross-validation with cosine annealing learning rate scheduling.

In [ ]:
import os
import glob
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import timm
from tqdm import tqdm

def resolve_dir(target_name):
    for root, dirs, files in os.walk("/kaggle/input"):
        if target_name in dirs:
            return os.path.join(root, target_name)
    for pat in [f"/kaggle/input/{target_name}", f"/kaggle/input/datasets/*/{target_name}", f"/kaggle/input/*/{target_name}"]:
        matches = glob.glob(pat)
        if matches:
            return matches[0]
    return f"/kaggle/input/datasets/barun2104/{target_name}"

def resolve_file(target_file):
    for root, dirs, files in os.walk("/kaggle/input"):
        if target_file in files:
            return os.path.join(root, target_file)
    for pat in [f"/kaggle/input/**/{target_file}", f"/kaggle/input/*/*/{target_file}"]:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return matches[0]
    return f"/kaggle/input/competitions/rsna-knee-abnormality-detection/{target_file}"

IMG_DIR = resolve_dir("rsna-knee-mri-processed-3d-volumes")
CSV_PATH = resolve_file("train_folds_with_pseudo.csv")
TRAIN_SERIES_PATH = resolve_file("train_series.csv")

TARGET_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Resolved IMG_DIR:", IMG_DIR)
print("Resolved CSV_PATH:", CSV_PATH)
print("Resolved TRAIN_SERIES_PATH:", TRAIN_SERIES_PATH)
print("Compute device:", device)


In [ ]:
df = pd.read_csv(CSV_PATH)
train_series_df = pd.read_csv(TRAIN_SERIES_PATH)
study_to_series_map = train_series_df.groupby("StudyInstanceUID")["SeriesInstanceUID"].apply(list).to_dict()
df["series_ids"] = df["StudyInstanceUID"].map(study_to_series_map)
df = df[df["series_ids"].notna() & (df["series_ids"].map(len) > 0)].reset_index(drop=True)

for col in TARGET_COLS:
    for cand in [f"pseudo_{col}", f"pseudo_{col.replace(' ', '_')}", f"{col}_pseudo"]:
        if cand in df.columns:
            df[col] = df[col].fillna(df[cand])
            break
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mean())

print(f"Total studies: {len(df):,}")


In [ ]:
CROP_FRAC = 0.8   # Top-scorer 0.8 centre field-of-view crop

def _crop(t):
    H, W = t.shape[-2], t.shape[-1]
    ch, cw = int(round(H * CROP_FRAC)), int(round(W * CROP_FRAC))
    y0, x0 = (H - ch) // 2, (W - cw) // 2
    return t[..., y0:y0 + ch, x0:x0 + cw]

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class WindowedKneeDataset(Dataset):
    def __init__(self, df, series_map, img_dir=IMG_DIR, is_train=True):
        self.df = df.reset_index(drop=True)
        self.series_map = series_map
        self.img_dir = img_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row["StudyInstanceUID"]
        series_ids = self.series_map.get(study_id, [])
        loaded_vols = []
        for s_id in series_ids:
            p = os.path.join(self.img_dir, f"{s_id}.npz")
            if os.path.exists(p):
                try:
                    with np.load(p) as npz:
                        k = "data" if "data" in npz else npz.files[0]
                        loaded_vols.append(npz[k])
                except Exception:
                    pass
        if len(loaded_vols) == 0:
            vol = np.zeros((24, 224, 224), dtype=np.float32)
        else:
            vol = np.concatenate(loaded_vols, axis=0)
            if vol.shape[0] != 24:
                idx_slices = np.linspace(0, vol.shape[0] - 1, 24).astype(int)
                vol = vol[idx_slices]
        
        windows = vol.reshape(8, 3, vol.shape[1], vol.shape[2]).astype(np.float32)
        if windows.max() > 1.0:
            windows /= 255.0
        tensor_w = torch.from_numpy(windows)
        
        # 0.8 centre crop from top-scorer
        tensor_w = _crop(tensor_w)
        
        tensor_w = F.interpolate(tensor_w, size=(224, 224), mode="bilinear", align_corners=False)
        tensor_w = (tensor_w - _MEAN) / _STD
        
        if self.is_train and np.random.rand() > 0.5:
            tensor_w = torch.flip(tensor_w, dims=[-1]) # Horizontal Flip
        y = torch.tensor(row[TARGET_COLS].values.astype(np.float32))
        return tensor_w, y


In [ ]:
class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=1024, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(
            nn.Linear(F_dim, 256),
            nn.Tanh(),
            nn.Dropout(drop),
            nn.Linear(256, n)
        )
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        flat = x.flatten(0, 1) # (B*K, 3, 224, 224)
        f = torch.cat([self.backbone(chunk) for chunk in flat.split(8)], dim=0)
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))

def build_model(pretrained=True):
    bb = timm.create_model("resnet50d", pretrained=pretrained, num_classes=0, in_chans=3)
    return RaptorClassifier(bb, F_dim=bb.num_features, n=len(TARGET_COLS))


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weights, gamma=1.5):
        super().__init__()
        self.register_buffer("w", torch.tensor(weights, dtype=torch.float32))
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        pt = targets * p + (1 - targets) * (1 - p)
        return (((1 - pt) ** self.gamma) * self.w * bce).mean()

# Calibrated top-scorer weights: Lateral OA (2.2), ACL/MCL (1.8), Meniscus (1.5)
WEIGHTS = [1.8, 1.8, 1.5, 1.5, 1.3, 2.2, 1.3, 1.5, 1.8, 1.8, 1.4, 1.4]
criterion = FocalLoss(WEIGHTS).to(device)

def compute_macro_auc(y_t, y_p):
    aucs = []
    for i in range(y_t.shape[1]):
        if len(np.unique(y_t[:, i])) > 1:
            aucs.append(roc_auc_score(y_t[:, i], y_p[:, i]))
    return float(np.mean(aucs)) if aucs else 0.5


In [ ]:
print("=== Starting Top-Scorer ResNet50d Raptor 5-Fold Training ===")
manual_df = df[df["fold"] != -1].reset_index(drop=True)
weak_df = df[df["fold"] == -1].reset_index(drop=True)
scaler = torch.amp.GradScaler("cuda")
fold_aucs = []

ACCUM_STEPS = 4

for fold_idx in range(5):
    print(f"\n--- Fold {fold_idx}/4 Training ---\n")
    model = build_model(pretrained=True).to(device)
    
    tr_expert = manual_df[manual_df["fold"] != fold_idx]
    tr_weak = weak_df.sample(n=150, random_state=42 + fold_idx)
    tr_df = pd.concat([tr_expert, tr_weak]).reset_index(drop=True)
    va_df = manual_df[manual_df["fold"] == fold_idx].reset_index(drop=True)
    
    tr_loader = DataLoader(WindowedKneeDataset(tr_df, study_to_series_map, is_train=True), batch_size=4, shuffle=True, num_workers=2)
    va_loader = DataLoader(WindowedKneeDataset(va_df, study_to_series_map, is_train=False), batch_size=4, shuffle=False, num_workers=2)
    
    head_params = list(model.norm.parameters()) + list(model.att.parameters()) + [model.clsW, model.clsb]
    head_ids = set(map(id, head_params))
    bb_params = [p for p in model.parameters() if id(p) not in head_ids]
    
    opt = torch.optim.AdamW([
        {"params": bb_params, "lr": 2e-5},
        {"params": head_params, "lr": 2e-4}
    ], weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5, eta_min=1e-6)
    
    best_auc = 0.0
    best_path = f"/kaggle/working/resnet50d_raptor_f{fold_idx}.pt"
    
    for ep in range(1, 6):
        model.train()
        opt.zero_grad()
        for step_idx, (x_b, y_b) in enumerate(tr_loader):
            x_b, y_b = x_b.to(device), y_b.to(device)
            with torch.amp.autocast("cuda"):
                loss = criterion(model(x_b), y_b) / ACCUM_STEPS
            scaler.scale(loss).backward()
            if (step_idx + 1) % ACCUM_STEPS == 0 or (step_idx + 1) == len(tr_loader):
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()
        sched.step()
        
        torch.cuda.empty_cache()
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for x_b, y_b in va_loader:
                x_b = x_b.to(device)
                with torch.amp.autocast("cuda"):
                    out = torch.sigmoid(model(x_b))
                preds.append(out.cpu().numpy())
                targets.append(y_b.numpy())
        val_auc = compute_macro_auc(np.vstack(targets), np.vstack(preds))
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), best_path)
            
    print(f"Fold {fold_idx} Complete | Best Val Macro AUC: {best_auc:.4f} -> Saved {best_path}")
    fold_aucs.append(best_auc)
    torch.cuda.empty_cache()

print(f"\n=== Top-Scorer ResNet50d Raptor 5-Fold Training Complete! Mean Val Macro AUC: {np.mean(fold_aucs):.4f} ===")
